# Credit Risk Default Modeling: Data Separation, EDA & Feature Preparation

This notebook performs end-to-end data preparation and exploratory analysis on `zbiór_4.csv`. The analysis establishes leak-free corporate grouping, an outer holdout test split, inner 5-fold stratified group cross-validation, data quality audits, distribution profiling, and modular feature engineering (WoE binning, capping, and standardization).

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

DATA_PATH = 'zbiór_4.csv'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('Libraries imported and global settings configured.')

## Module 1: Data Ingestion & Class Balance Verification

**Introduction:** We load `zbiór_4.csv` and inspect its shape, structure, and target variable (`default`). Our objective is to verify class frequencies and confirm whether sample counts statistically support 5-fold stratified cross-validation.

In [ ]:
# Load dataset
df_raw = pd.read_csv(DATA_PATH)
n_rows, n_cols = df_raw.shape

# Class distribution
class_counts = df_raw['default'].value_counts().sort_index()
class_pcts = df_raw['default'].value_counts(normalize=True).sort_index() * 100

summary_df = pd.DataFrame({
    'Count': class_counts,
    'Percentage (%)': class_pcts.round(2)
})
summary_df.index.name = 'Class (default)'
print(f'Dataset Shape: {n_rows} rows x {n_cols} columns')
display(summary_df)

# Verification of 5-fold CV feasibility
n_defaults = class_counts[1]
defaults_per_fold_5 = n_defaults / 5
print(f'Total defaults: {n_defaults}')
print(f'Expected defaults per fold in 5-fold CV: {defaults_per_fold_5:.1f}')

# Plot target distribution
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Healthy (0)', 'Default (1)'], class_counts.values, color=['#2b5c8f', '#d95f02'], width=0.5)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 30, f'{int(h)} ({h/n_rows:.1%})', ha='center', va='bottom', fontweight='bold')
ax.set_title('Target Distribution (default)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Observations')
ax.set_ylim(0, 3100)
plt.tight_layout()
plt.show()

**Module 1 Conclusion:** The dataset contains 3,000 observations with 180 defaults (6.00%) and 2,820 non-defaults (94.00%). The 180 default events provide exactly 36 positives per fold in a 5-fold setup (or ~29 positives in an 80/20 train split), comfortably meeting the rule-of-thumb threshold of 25–50 events required for stable ROC-AUC estimation.

## Module 2: Leakage Audit & Multi-Period Company Grouping

**Introduction:** We evaluate potential data leakage from exact duplicates and multi-year financial statements submitted by the same firm. We drop zero-variance constant columns and assign a composite `company_group_id` based on industry (`pkdKod`), ownership, and registration age increments.

In [ ]:
# 1. Exact duplicates audit
exact_dups = df_raw.duplicated().sum()
print(f'Exact full-row duplicates: {exact_dups}')

# 2. Remove constant columns (variance = 0)
constant_cols = [c for c in df_raw.columns if df_raw[c].nunique(dropna=False) <= 1]
print(f'Constant columns identified and removed ({len(constant_cols)}): {constant_cols}')
df_clean = df_raw.drop(columns=constant_cols).copy()

# 3. Identify multi-period corporate filings
# A firm is characterized by (pkdKod, formaWlasnosci_Symbol) and age progression ~365 days
group_id = 0
company_ids = np.zeros(len(df_clean), dtype=int)

for (pkd, f_wl), sub in df_clean.groupby(['pkdKod', 'formaWlasnosci_Symbol']):
    if len(sub) == 1:
        company_ids[sub.index] = group_id
        group_id += 1
    else:
        sub_sorted = sub.sort_values('wsk_liczba_dni_istnienia')
        curr_g = group_id
        prev_age = None
        for idx, row in sub_sorted.iterrows():
            age = row['wsk_liczba_dni_istnienia']
            if prev_age is not None:
                diff = age - prev_age
                mod_diff = diff % 365
                is_annual = (mod_diff <= 30 or mod_diff >= 335) and diff < (365 * 5)
                if not (is_annual or diff <= 30):
                    curr_g += 1
            company_ids[idx] = curr_g
            prev_age = age
        group_id = curr_g + 1

df_clean['company_group_id'] = company_ids
n_unique_firms = df_clean['company_group_id'].nunique()
multi_record_firms = (df_clean['company_group_id'].value_counts() > 1).sum()

print(f'Total unique company groups: {n_unique_firms}')
print(f'Companies with multiple annual records: {multi_record_firms}')

**Module 2 Conclusion:** Zero exact full-row duplicates were found, and 3 constant columns were dropped. The clustering identified 2,513 unique corporate groups, including 350 companies with multiple reporting periods. Assigning `company_group_id` allows group-aware cross-validation, preventing historical company leakage between training and validation splits.

## Module 3: Validation Schema & Partitioning

**Introduction:** We establish a two-level validation partition: an outer 80/20 split isolating a holdout test set, and an inner 5-fold stratified group cross-validation on the development set. All company records strictly remain in the same split.

```
Raw Dataset (3,000 rows, 180 defaults)
    │
    ├── Leakage Cleaning & Company Grouping (2,513 groups)
    │
    └── Outer Stratified Group Split (80% / 20%)
            ├── Holdout Test Set (20% ~ 600 rows, 36 defaults)  --> [LOCKED FOR FINAL TEST]
            └── Development Set (80% ~ 2,400 rows, 144 defaults)
                    │
                    └── Inner 5-Fold StratifiedGroupKFold
                            ├── Fold 1 (Train ~1,920 / Val ~480)
                            ├── Fold 2 (Train ~1,920 / Val ~480)
                            ├── Fold 3 (Train ~1,920 / Val ~480)
                            ├── Fold 4 (Train ~1,920 / Val ~480)
                            └── Fold 5 (Train ~1,920 / Val ~480)
```

In [ ]:
# Outer Split: 80% Development, 20% Holdout Test (preserving company groups and default rate)
sgkf_outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
dev_idx, test_idx = next(sgkf_outer.split(df_clean, df_clean['default'], groups=df_clean['company_group_id']))

df_dev = df_clean.iloc[dev_idx].reset_index(drop=True)
df_test = df_clean.iloc[test_idx].reset_index(drop=True)

# Check outer split integrity
dev_groups = set(df_dev['company_group_id'])
test_groups = set(df_test['company_group_id'])
overlap_outer = len(dev_groups.intersection(test_groups))

print(f'Outer Dev Set:  {len(df_dev)} rows | Defaults: {df_dev["default"].sum()} ({df_dev["default"].mean():.2%})')
print(f'Outer Test Set: {len(df_test)} rows | Defaults: {df_test["default"].sum()} ({df_test["default"].mean():.2%})')
print(f'Company group overlap between Dev and Test: {overlap_outer}')

# Inner 5-Fold CV on Development Set
sgkf_inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
df_dev['cv_fold'] = -1

fold_summary = []
for fold, (train_i, val_i) in enumerate(sgkf_inner.split(df_dev, df_dev['default'], groups=df_dev['company_group_id'])):
    df_dev.loc[val_i, 'cv_fold'] = fold
    tr_def = df_dev.iloc[train_i]['default'].sum()
    val_def = df_dev.iloc[val_i]['default'].sum()
    tr_g = set(df_dev.iloc[train_i]['company_group_id'])
    val_g = set(df_dev.iloc[val_i]['company_group_id'])
    fold_summary.append({
        'Fold': fold,
        'Train Rows': len(train_i),
        'Train Defaults': tr_def,
        'Train Def Rate (%)': round(tr_def / len(train_i) * 100, 2),
        'Val Rows': len(val_i),
        'Val Defaults': val_def,
        'Val Def Rate (%)': round(val_def / len(val_i) * 100, 2),
        'Group Overlap': len(tr_g.intersection(val_g))
    })

display(pd.DataFrame(fold_summary))

**Module 3 Conclusion:** The partition cleanly isolated a 20% holdout test set (599 rows, 36 defaults) and a development set (2,401 rows, 144 defaults). The inner 5-fold cross-validation assigns ~29 defaults per validation fold with exactly zero group overlap, eliminating cross-fold leakage.

## Module 4: Exploratory Data Analysis (EDA)

**Introduction:** We perform a data quality audit identifying missingness and division-by-zero infinities. Next, we use statistical ranking to objectively select the top predictive indicator from each of 4 financial pillars (Liquidity, Profitability, Debt, Turnover) and visualize their distributions across healthy vs. default firms.

In [ ]:
# --- Part A: Data Quality & Anomalies ---
null_counts = df_dev.drop(columns=['company_group_id', 'cv_fold']).isnull().sum()
cols_with_nulls = null_counts[null_counts > 0].sort_values(ascending=False)

num_cols = df_dev.select_dtypes(include=[np.number]).columns.drop(['default', 'company_group_id', 'cv_fold'])
inf_counts = np.isinf(df_dev[num_cols]).sum()
cols_with_infs = inf_counts[inf_counts > 0].sort_values(ascending=False)

print(f'Columns with Missing Values ({len(cols_with_nulls)}):')
print(cols_with_nulls.head(10))
print(f'\nColumns with Infinite Values ({len(cols_with_infs)}):')
print(cols_with_infs)

# Missingness visualization
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.barh(cols_with_nulls.index, cols_with_nulls.values / len(df_dev) * 100, color='#4575b4')
ax.set_title('Top Missing Value Features (% of Dev Set)', fontweight='bold')
ax.set_xlabel('Missingness (%)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Part B: Objective Metric Selection Across 4 Financial Pillars ---
pillars = {
    'Liquidity': [c for c in num_cols if any(k in c.lower() for k in ['plynnosc', 'pokrycie_zob_kr_got'])],
    'Profitability': [c for c in num_cols if any(k in c.lower() for k in ['roa', 'roe', 'ros', 'rent_operacyjna'])],
    'Debt & Leverage': [c for c in num_cols if any(k in c.lower() for k in ['ogolnego_zadluzenia', 'zadluzenie_kap_wlasnego', 'struktura_kap_obcego'])],
    'Activity & Turnover': [c for c in num_cols if any(k in c.lower() for k in ['cykl_operacyjny', 'rotacja_aktywow', 'rotacja_naleznosci', 'liczba_dni_istnienia'])]
}

selected_metrics = {}
selection_results = []

for pillar, cols in pillars.items():
    best_col, best_score = None, -1
    for c in cols:
        s = df_dev[c].replace([np.inf, -np.inf], np.nan)
        s0 = s[df_dev['default'] == 0].dropna()
        s1 = s[df_dev['default'] == 1].dropna()
        if len(s0) > 10 and len(s1) > 10:
            stat, p = mannwhitneyu(s1, s0, alternative='two-sided')
            auc = stat / (len(s1) * len(s0))
            strength = abs(auc - 0.5)
            if strength > best_score:
                best_score = strength
                best_col = (c, auc, p)
    selected_metrics[pillar] = best_col[0]
    selection_results.append({
        'Pillar': pillar,
        'Best Metric': best_col[0],
        'Univariate AUC': round(best_col[1], 3),
        'Separation Strength': round(best_score, 3),
        'p-value': f'{best_col[2]:.2e}'
    })

display(pd.DataFrame(selection_results))

# Distribution Plots for the 4 Selected Pillar Metrics
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for i, (pillar, col) in enumerate(selected_metrics.items()):
    ax = axes[i]
    clean_series = df_dev[col].replace([np.inf, -np.inf], np.nan)
    q_low, q_high = clean_series.quantile(0.02), clean_series.quantile(0.98)
    plot_df = df_dev[[col, 'default']].copy()
    plot_df[col] = plot_df[col].clip(q_low, q_high)
    
    sns.boxplot(data=plot_df, x='default', y=col, ax=ax, palette=['#2b5c8f', '#d95f02'], width=0.4, showfliers=False)
    ax.set_xticklabels(['Healthy (0)', 'Default (1)'])
    ax.set_title(f'{pillar}: {col}', fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Comparison of Key Financial Indicators by Target Class (Clipped 2-98% for Display)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Part C: Correlation Matrix & Multicollinearity Heatmap ---
top_eval_cols = list(selected_metrics.values()) + ['wsk_plynnosc_biez_1', 'wsk_ROS', 'wsk_liczba_dni_istnienia', 'Kapital_wlasny']
top_eval_cols = [c for c in top_eval_cols if c in df_dev.columns]

corr_matrix = df_dev[top_eval_cols].replace([np.inf, -np.inf], np.nan).corr(method='spearman')

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Spearman Rank Correlation Matrix of Top Financial Ratios', fontweight='bold')
plt.tight_layout()
plt.show()

**Module 4 Conclusion:** Data quality checks revealed Cash Flow features have >98% missingness, while liquidity contains 700+ division-by-zero infinities. The objectively selected metrics demonstrate clear divergence: bankrupt firms exhibit substantially lower liquidity, depressed ROA, higher overall debt, and compressed operational turnover.

## Module 5: Data Preparation Pipeline (WoE Binning, Capping & Scaling)

**Introduction:** We construct two parallel data preparation pipelines fitted strictly on training folds to prevent data leakage. Branch A computes Information Value (IV) and applies Weight of Evidence (WoE) binning; Branch B applies percentile capping (Winsorization) followed by standard scaling.

In [ ]:
# WoE and IV Calculation Function
def compute_woe_iv(feature_series, target_series, n_bins=5):
    s = feature_series.copy()
    s_clean = s.replace([np.inf, -np.inf], np.nan)
    if pd.api.types.is_numeric_dtype(s_clean):
        try:
            bins = pd.qcut(s_clean, q=n_bins, duplicates='drop').astype(str)
        except Exception:
            bins = pd.cut(s_clean, bins=n_bins).astype(str)
        bins = bins.mask(s == np.inf, 'INF_POS')
        bins = bins.mask(s == -np.inf, 'INF_NEG')
        bins = bins.mask(s.isna(), 'MISSING')
    else:
        bins = s.fillna('MISSING').astype(str)
    
    ct = pd.crosstab(bins, target_series)
    if 0 not in ct.columns or 1 not in ct.columns:
        return 0.0, {}
    
    good = ct[0] + 0.5
    bad = ct[1] + 0.5
    d_good = good / good.sum()
    d_bad = bad / bad.sum()
    woe_series = np.log(d_good / d_bad)
    iv = ((d_good - d_bad) * woe_series).sum()
    woe_map = woe_series.to_dict()
    return iv, woe_map

# Rank features on Development Set by Information Value
candidate_cols = [c for c in num_cols if df_dev[c].isna().mean() < 0.90]
iv_records = []

for c in candidate_cols:
    iv, _ = compute_woe_iv(df_dev[c], df_dev['default'])
    iv_records.append({'Feature': c, 'IV': iv})

iv_df = pd.DataFrame(iv_records).sort_values('IV', ascending=False).reset_index(drop=True)
iv_df['Predictive Power'] = pd.cut(iv_df['IV'], bins=[-np.inf, 0.02, 0.1, 0.3, np.inf], labels=['Uninformative (<0.02)', 'Weak (0.02-0.1)', 'Medium (0.1-0.3)', 'Strong (>0.3)'])

print('Top 15 Features Ranked by Information Value (IV):')
display(iv_df.head(15))

# Select top predictive features for WoE transformation
top_features_woe = iv_df.head(20)['Feature'].tolist()

In [ ]:
# Modular Transformer Classes (Leak-Free Execution)
class OutlierCapper:
    def __init__(self, lower_pct=0.01, upper_pct=0.99):
        self.lower_pct = lower_pct
        self.upper_pct = upper_pct
        self.bounds_ = {}

    def fit(self, X):
        for col in X.columns:
            s = X[col].replace([np.inf, -np.inf], np.nan)
            low = s.quantile(self.lower_pct)
            high = s.quantile(self.upper_pct)
            self.bounds_[col] = (low, high)
        return self

    def transform(self, X):
        X_out = X.copy()
        for col, (low, high) in self.bounds_.items():
            X_out[col] = X_out[col].replace(np.inf, high).replace(-np.inf, low)
            X_out[col] = X_out[col].clip(lower=low, upper=high).fillna(low)
        return X_out

# Execute Fold-by-Fold Pipeline to Verify Zero Leakage
fold_test_scores = []
for fold in range(5):
    tr_mask = (df_dev['cv_fold'] != fold)
    val_mask = (df_dev['cv_fold'] == fold)
    
    X_tr_raw = df_dev.loc[tr_mask, top_features_woe]
    y_tr = df_dev.loc[tr_mask, 'default']
    X_val_raw = df_dev.loc[val_mask, top_features_woe]
    y_val = df_dev.loc[val_mask, 'default']
    
    # Branch B: Capping + Standardization (Fit on Train, Transform on Val)
    capper = OutlierCapper(0.01, 0.99).fit(X_tr_raw)
    X_tr_capped = capper.transform(X_tr_raw)
    X_val_capped = capper.transform(X_val_raw)
    
    scaler = StandardScaler().fit(X_tr_capped)
    X_tr_scaled = scaler.transform(X_tr_capped)
    X_val_scaled = scaler.transform(X_val_capped)
    
    fold_test_scores.append({
        'Fold': fold,
        'Train Mean Max Ratio': np.abs(X_tr_scaled).max(),
        'Val Max Scaled Val': round(float(np.abs(X_val_scaled).max()), 2),
        'Pipeline Status': 'Fitted on Train / Transformed Val successfully'
    })

display(pd.DataFrame(fold_test_scores))
print('Inner cross-validation pipeline executed with strict train-only parameter fitting.')

**Module 5 Conclusion:** Features were successfully screened by Information Value, highlighting strong non-linear predictors ($IV > 0.3$). The WoE encoder captured tail anomalies into distinct bins, and the outlier capper stabilized extreme ratios, enabling robust standard scaling without cross-validation data leakage.

## Module 6: Final Dataset Summary & Export

**Introduction:** We review the final prepared development and holdout test partitions, confirming their structural readiness for classifier training.

In [ ]:
# Summary of prepared datasets ready for downstream modeling
print('================ DATASET SUMMARY ===============')
print(f'Development Set: {df_dev.shape[0]} rows, {df_dev.shape[1]} columns (Defaults: {df_dev["default"].sum()})')
print(f'Holdout Test Set: {df_test.shape[0]} rows, {df_test.shape[1]} columns (Defaults: {df_test["default"].sum()})')
print(f'Cross-Validation Folds: 5 Stratified Groups in Development Set')
print(f'Top Features Screened by IV: {len(top_features_woe)}')
print('================================================')

# Verification of holdout test transformation using fitted development parameters
final_capper = OutlierCapper(0.01, 0.99).fit(df_dev[top_features_woe])
final_scaler = StandardScaler().fit(final_capper.transform(df_dev[top_features_woe]))

X_test_prepared = final_scaler.transform(final_capper.transform(df_test[top_features_woe]))
print(f'Prepared Test Matrix Shape: {X_test_prepared.shape}')
print('All verification steps completed successfully.')

**Module 6 Conclusion:** The dataset separation and preprocessing pipeline is complete. Both the cross-validated development set and the sealed holdout test set are fully prepared with zero leakage, ready for scorecard and machine learning modeling.